In [37]:
import pandas as pd

from src.metrics.process_data import metrics_processor
from src.utils import clean_text

In [24]:
!python -m spacy download es_core_news_sm

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     --------------------------------------- 12.9/12.9 MB 73.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
df = pd.read_excel("comparacion_capa2.xlsx", sheet_name="Matriz comparativa")

In [26]:
df.iloc[12]

#                                                                                                                NaN
Sección                                                            TEXTO ADAPTADO (SALIDA DE CADA HERRAMIENTA DE LF)
Original                                                           Información sobre protección de datos\nEste cu...
FACILE                                                             Información sobre protección de datos.\nSe ha ...
Placea                                                             INFORMACIÓN SOBRE PROTECCIÓN DE DATOS\n\nEste ...
Asistente de lectura fácil “ANTONIO GONZALES CRESPO”               Información sobre protección de datos\n\nEste ...
Asistente de lectura fácil “Mark Jonathan Camacho Escatel”:        Información sobre protección de datos\n\nEste ...
Gemini 3.1 Pro                                                     Información sobre la protección de tus datos\n...
GPT 5.4 Think                                                   

In [27]:
df.iloc[12]["Original"]

'Información sobre protección de datos\nEste cuestionario ha sido diseñado de forma que se preserve totalmente su carácter anónimo. No obstante, consideramos oportuno informarte en relación con nuestra política de privacidad, conforme a lo establecido en la normativa de protección de datos de carácter personal. Si accedes a cumplimentar el cuestionario te informamos de que consientes que tus datos sean tratados por la Universidad Politécnica de Madrid (Grupo de Ingeniería Ontológica), responsable de este tratamiento, con la finalidad relacionada con la investigación ya señalada. También te informamos de que puedes acceder, rectificar y suprimir los datos, así como ejercer otros derechos, en los términos que se indican en la información adicional, disponible en el siguiente enlace: "https://oeg.fi.upm.es/protecciondatos.html".'

In [28]:
new_df = df.iloc[12].to_frame(name="prediction_text")
new_df.rename(columns={"12":"prediction_text"}, inplace=True)
new_df["original_text"] = df.iloc[12]["Original"]
new_df["simplified_text"] = df.iloc[12]["Original"]
new_df["document_id"] = new_df.index
new_df["original_sentence_id"] = new_df.index

In [29]:
new_df.columns

Index(['prediction_text', 'original_text', 'simplified_text', 'document_id',
       'original_sentence_id'],
      dtype='str')

In [34]:
df_to_calculate_metric = new_df[3:]

In [38]:
# 2. Aplicar limpieza
for col in ['prediction_text', 'original_text', 'simplified_text']:
    if col in df_to_calculate_metric.columns:
        df_to_calculate_metric[col] = df_to_calculate_metric[col].apply(clean_text)

# 3. Filtro de longitud y exportación
df_to_calculate_metric = df_to_calculate_metric[df_to_calculate_metric['prediction_text'].str.len() > 5]

# Guardamos el archivo limpio para que MetricsProcessor lo use sin errores
df_to_calculate_metric.to_excel('clean.xlsx', index=False)

print(f"Limpieza finalizada. Filas listas para procesar: {len(df_to_calculate_metric)}")
df_to_calculate_metric.head()


Limpieza finalizada. Filas listas para procesar: 11


,prediction_text,original_text,simplified_text,document_id,original_sentence_id
FACILE,Información sobre protección de datos. Se ha d...,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,FACILE,FACILE
Placea,INFORMACIÓN SOBRE PROTECCIÓN DE DATOS Este cue...,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Placea,Placea
Asistente de lectura fácil “ANTONIO GONZALES CRESPO”,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Asistente de lectura fácil “ANTONIO GONZALES C...,Asistente de lectura fácil “ANTONIO GONZALES C...
Asistente de lectura fácil “Mark Jonathan Camacho Escatel”:,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Asistente de lectura fácil “Mark Jonathan Cama...,Asistente de lectura fácil “Mark Jonathan Cama...
Gemini 3.1 Pro,Información sobre la protección de tus datos E...,Información sobre protección de datos Este cue...,Información sobre protección de datos Este cue...,Gemini 3.1 Pro,Gemini 3.1 Pro


In [39]:
metric = metrics_processor(df_to_calculate_metric)
metric.save_consolidated_report("metrics.xlsx")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5555.22it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5185.32it/s]
c:\Users\prestamo_admin\Documents\01_repo\02_legal_analisys\.venv\Lib\site-packages\openpyxl\worksheet\header_f

Reporte consolidado guardado en: metrics.xlsx
